Import Required Libraries


In [ ]:
import numpy as np
import tensorflow as tf
import random

seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)
random.seed(seed)

In [ ]:
!pip install lifelines
!pip install gseapy
!pip install decoupler
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import gseapy as gp
import time
import textwrap
import decoupler as dc

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold

from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from sklearn.decomposition import PCA

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score, confusion_matrix

from scipy.stats import ranksums
from statsmodels.stats.multitest import multipletests

Data Loading (Gene Expression)

In [ ]:
rna_url = "https://tcga-xena-hub.s3.us-east-1.amazonaws.com/download/TCGA.PRAD.sampleMap/HiSeqV2.gz"

rna = pd.read_csv(rna_url, sep="\t", index_col=0)
print("Original RNA shape:", rna.shape)

Data Pre Processing (Gene Expression)

In [ ]:
rna = rna.T
rna.index = rna.index.str[:12]
rna = rna[~rna.index.duplicated(keep="first")]
print("RNA after transpose:", rna.shape)

In [ ]:
threshold = int(0.2 * rna.shape[0])
rna = rna.loc[:, (rna > 0).sum(axis=0) > threshold]
print("After zero filtering:", rna.shape)

In [ ]:
gene_variance = rna.var(axis=0)
sorted_variance = (gene_variance.sort_values(ascending=False).values)

plt.figure(figsize=(7, 5))

plt.plot(sorted_variance,linewidth=2,color="#4C72B0")

plt.axhline(y=1.0,linestyle="--",linewidth=2,color="#9A9A9A",label="Variance threshold = 1.0")

plt.xlabel("Gene Rank")
plt.ylabel("Gene Variance")
plt.title("Variance Distribution of Genes")

plt.grid(axis='y',alpha=0.08)
plt.ylim(0,max(sorted_variance) * 1.05)
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
threshold = 1.0
selected_genes = gene_variance[gene_variance > threshold]
print("Number of genes with variance > 1.0:", len(selected_genes))
rna = rna[selected_genes.index]
print(rna.shape)
rna.head()

Data Loading & Data Pre Processing (Survival Data)

In [ ]:
clinical_url = "https://tcga-xena-hub.s3.us-east-1.amazonaws.com/download/TCGA.PRAD.sampleMap%2FPRAD_clinicalMatrix"
clinical = pd.read_csv(clinical_url, sep="\t", index_col=0)
clinical["PatientID"] = clinical.index.str[:12]
clinical = clinical.set_index("PatientID")
clinical = clinical.groupby(clinical.index).first()
print("Clinical shape:", clinical.shape)

In [ ]:
def yesno(x):
    if str(x).upper()=="YES":
        return 1
    if str(x).upper()=="NO":
        return 0
    return np.nan

clinical["rec1"] = clinical["biochemical_recurrence"].apply(yesno)
clinical["rec2"] = clinical["new_tumor_event_after_initial_treatment"].apply(yesno)
clinical["rec3"] = clinical["tumor_progression_post_ht"].apply(yesno)
clinical["recurrence"] = clinical[["rec1","rec2","rec3"]].max(axis=1)
print("Recurrence events:", clinical["recurrence"].sum())

In [ ]:
t1 = pd.to_numeric(clinical["days_to_first_biochemical_recurrence"], errors="coerce")
t2 = pd.to_numeric(clinical["days_to_new_tumor_event_after_initial_treatment"], errors="coerce")
t3 = pd.to_numeric(clinical["days_to_last_followup"], errors="coerce")

event_time = pd.concat([t1, t2], axis=1).min(axis=1)
event_time = event_time.fillna(t3)

clinical["time"] = np.where(clinical["recurrence"] == 1,event_time,t3)

clinical = clinical.dropna(subset=["time","recurrence"])
clinical = clinical[clinical["time"] > 0]

In [ ]:
print(clinical.index.nunique(), len(clinical))

Align RNA and Survival data

In [ ]:
common = rna.index.intersection(clinical.index)
print("Number of common patients:", len(common))
rna_final = rna.loc[common].copy()
survival = clinical.loc[common, ["time","recurrence"]].copy()
print("RNA final shape:", rna_final.shape)
print("Survival shape:", survival.shape)

In [ ]:
assert all(rna_final.index == survival.index)

Latent Feature Extraction Using Autoencoder

In [ ]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
K.clear_session()

scaler = StandardScaler()
X_full = scaler.fit_transform(rna_final)

input_dim = X_full.shape[1]
inp = Input(shape=(input_dim,))
x = Dense(256, activation='tanh',kernel_regularizer=regularizers.l2(1e-4))(inp)
bottleneck = Dense(100, activation='tanh')(x)
x = Dense(256, activation='tanh')(bottleneck)
out = Dense(input_dim, activation='linear')(x)
autoencoder = Model(inp, out)
encoder = Model(inp, bottleneck)
autoencoder.compile(optimizer='adam', loss='mse')
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = autoencoder.fit(
    X_full, X_full,
    epochs=15,
    batch_size=32,
    validation_split=0.2,
    shuffle=True,
    callbacks=[early_stop],
    verbose=1
)

Z = encoder.predict(X_full, verbose=0)
latent_df = pd.DataFrame(Z, index=rna_final.index)

cox_df = latent_df.copy()
cox_df["time"] = survival["time"].values
cox_df["event"] = survival["recurrence"].values

cph = CoxPHFitter(penalizer=0.1)
cph.fit(cox_df, duration_col="time", event_col="event")

print("\nFinal Cox Model Trained")

encoder.save("final_encoder.h5")
autoencoder.save("final_autoencoder.h5")

print("Models saved successfully")

In [ ]:
plt.figure(figsize=(6,4))

plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('Autoencoder Training Loss')

plt.legend(loc="upper right", frameon=False)
plt.grid(alpha=0.08)

plt.tight_layout()
plt.show()

In [ ]:
autoencoder.summary()

In [ ]:
print(latent_df.head())
print(latent_df.shape)

Survival Modeling


In [ ]:
pvals = []
valid_cols = []

for col in latent_df.columns:

    df = pd.DataFrame({
        col: latent_df[col],
        "time": survival["time"],
        "event": survival["recurrence"]
    }).dropna()

    try:
        cph = CoxPHFitter()
        cph.fit(df, duration_col="time", event_col="event")

        p = cph.summary.loc[col, "p"]
        pvals.append(p)
        valid_cols.append(col)

    except:
        pvals.append(1)
        valid_cols.append(col)

fdr = multipletests(pvals, method="fdr_bh")[1]

significant_nodes = [col for col, f in zip(valid_cols, fdr) if f < 0.05]
print("Significant nodes:", len(significant_nodes))

In [ ]:
latent_sig = latent_df[significant_nodes]

scaler = StandardScaler()
X = scaler.fit_transform(latent_sig)

k_range = range(2, 8)
sil_scores = []
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = kmeans.fit_predict(X)
    sil_scores.append(silhouette_score(X, labels))

k_vals = list(k_range)
best_k = k_vals[np.argmax(sil_scores)]
best_score = max(sil_scores)

print("Best K:", best_k)

plt.figure(figsize=(6, 4))

plt.plot(k_vals, sil_scores, marker='o')
plt.axvline(best_k, linestyle='--', alpha=0.7)
plt.scatter(best_k, best_score, s=80)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Optimal Cluster Selection")

plt.xticks(k_vals)
plt.grid(alpha=0.2)

plt.tight_layout()
plt.show()

Identification of Recurrence-Risk Subgroups


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
latent_sig_scaled = scaler.fit_transform(latent_sig)

kmeans = KMeans(n_clusters=2, random_state=42, n_init=20)
clusters = kmeans.fit_predict(latent_sig_scaled)

print(pd.Series(clusters).value_counts())

In [ ]:
risk_df = survival.copy()
risk_df["cluster"] = clusters

group0_mean = risk_df[risk_df["cluster"] == 0]["time"].mean()
group1_mean = risk_df[risk_df["cluster"] == 1]["time"].mean()

if group0_mean > group1_mean:
    label_map = {0: "Low Risk", 1: "High Risk"}
else:
    label_map = {1: "Low Risk", 0: "High Risk"}

print("Cluster labels:", label_map)

kmf = KaplanMeierFitter()
plt.figure(figsize=(7,5))

for label in ["Low Risk", "High Risk"]:
    for c, name in label_map.items():
        if name == label:
            mask = risk_df["cluster"] == c

            kmf.fit(
                durations=risk_df["time"][mask],
                event_observed=risk_df["recurrence"][mask],
                label=name
            )

            kmf.plot_survival_function(ci_show=True)

g0 = risk_df[risk_df["cluster"] == 0]
g1 = risk_df[risk_df["cluster"] == 1]

lr = logrank_test(
    g0["time"], g1["time"],
    event_observed_A=g0["recurrence"],
    event_observed_B=g1["recurrence"]
)

p_value = lr.p_value
print("Log-rank p-value:", p_value)

plt.xlabel("Time (Days)")
plt.ylabel("Survival Probability")
plt.title("Kaplan–Meier Survival Analysis")

plt.legend(frameon=False)

plt.text(
    0.6, 0.3,
    f"Log-rank p = {p_value:.2e}",
    transform=plt.gca().transAxes,
    fontsize=10
)
plt.xlim(0, 4000)

plt.grid(alpha=0.08)
plt.tight_layout()

plt.savefig("KM_plot.png", dpi=400)

plt.show()

Differential Gene Expression Analysis

In [ ]:
rna_deg = rna_final.loc[latent_df.index].copy()
rna_deg["cluster"] = clusters

In [ ]:
group0 = rna_deg[rna_deg["cluster"] == 0].drop("cluster", axis=1)
group1 = rna_deg[rna_deg["cluster"] == 1].drop("cluster", axis=1)

print("Group 0:", group0.shape)
print("Group 1:", group1.shape)

In [ ]:
deg_results = []
for gene in group0.columns:

    try:
        stat, p = ranksums(group0[gene], group1[gene])

        mean0 = group0[gene].mean()
        mean1 = group1[gene].mean()

        logFC = mean1 - mean0

        deg_results.append([gene, logFC, p])

    except Exception as e:
      print(f"Error in gene {gene}: {e}")
      deg_results.append([gene, 0, 1])

deg_df = pd.DataFrame(deg_results, columns=["gene", "logFC", "pvalue"])
deg_df["FDR"] = multipletests(deg_df["pvalue"], method="fdr_bh")[1]
deg_df = deg_df.sort_values("FDR")
deg_filtered = deg_df[
    (deg_df["FDR"] < 0.05) &
    (deg_df["logFC"].abs() > 1)
]

print("Total genes:", len(deg_df))
print("Significant genes:", len(deg_filtered))
print("\nTop significant genes:")
print(deg_filtered.head(10))


Functional Enrichment Analysis


In [ ]:
all_list=deg_filtered["gene"].tolist()

In [ ]:
go_all = gp.enrichr(
    gene_list=all_list,
    organism="human",
    gene_sets="GO_Biological_Process_2021"
)

In [ ]:
go_all_df = go_all.results
go_all_sig = go_all_df[go_all_df["Adjusted P-value"] < 0.05].sort_values("Adjusted P-value")

In [ ]:
print(go_all_sig[["Term", "Adjusted P-value"]].head(5))

In [ ]:
kegg_all = gp.enrichr(
    gene_list=all_list,
    organism="human",
    gene_sets="KEGG_2021_Human"
)

In [ ]:
kegg_all_df = kegg_all.results
kegg_all_sig = kegg_all_df[kegg_all_df["Adjusted P-value"] < 0.05].sort_values("Adjusted P-value")

In [ ]:
print(kegg_all_sig[["Term", "Adjusted P-value"]].head(5))

In [ ]:
go_plot = go_all_sig.head(10).copy()

go_plot["Term"] = go_plot["Term"].str.replace(
    r"\s*\(GO:\d+\)", "", regex=True
)

go_plot["Term"] = go_plot["Term"].apply(
    lambda x: "\n".join(textwrap.wrap(x, width=35))
)

go_plot["-log10(p)"] = -np.log10(
    go_plot["Adjusted P-value"]
)

go_plot = go_plot[::-1]

fig, ax = plt.subplots(figsize=(12, 7))

ax.barh(
    go_plot["Term"],
    go_plot["-log10(p)"],
    color="#4C84B5",
    edgecolor="black",
    linewidth=0.5,
    height=0.6
)

ax.set_title(
    "GO Biological Processes",
    fontweight="bold",
    pad=12
)

ax.set_xlabel(
    "-log10(Adjusted P-value)",
    fontweight="bold"
)

ax.set_ylabel(
    "GO Biological Processes",
    fontweight="bold"
)

for label in ax.get_xticklabels():
    label.set_fontweight('bold')

for label in ax.get_yticklabels():
    label.set_fontweight('bold')

ax.grid(
    axis='x',
    linestyle='--',
    alpha=0.4
)

#ax.spines['top'].set_visible(False)
#ax.spines['right'].set_visible(False)

plt.subplots_adjust(left=0.50)
plt.show()

In [ ]:
kegg_plot = kegg_all_sig.head(10).copy()

kegg_plot["-log10(p)"] = -np.log10(
    kegg_plot["Adjusted P-value"]
)

kegg_plot = kegg_plot[::-1]

fig, ax = plt.subplots(figsize=(12, 7))

ax.barh(
    kegg_plot["Term"],
    kegg_plot["-log10(p)"],
    color="#4C84B5",
    edgecolor="black",
    linewidth=0.5,
    height=0.6
)

ax.set_title(
    "KEGG Pathways",
    fontweight="bold",
    pad=12
)

ax.set_xlabel(
    "-log10(Adjusted P-value)",
    fontweight="bold"
)

ax.set_ylabel(
    "KEGG Pathways",
    fontweight="bold"
)

for label in ax.get_xticklabels():
    label.set_fontweight('bold')

for label in ax.get_yticklabels():
    label.set_fontweight('bold')

ax.grid(
    axis='x',
    linestyle='--',
    alpha=0.4
)

#ax.spines['top'].set_visible(False)
#ax.spines['right'].set_visible(False)

plt.subplots_adjust(left=0.50)
plt.show()

Transcription Factor Activity and Master Regulators

In [ ]:
net = dc.op.dorothea(
    organism="human",
    levels=["A","B"]
)
net.head()

In [ ]:
common_genes = list(set(rna_deg.columns) & set(net["target"]))
expr = rna_deg[common_genes].copy()

acts, pvals = dc.mt.ulm(
    data=expr,
    net=net,
    verbose=True
)

print("TF activity shape:", acts.shape)

In [ ]:
acts_copy = acts.copy()

acts_copy["cluster"] = clusters

group0 = acts_copy[acts_copy["cluster"] == 0].drop("cluster", axis=1)
group1 = acts_copy[acts_copy["cluster"] == 1].drop("cluster", axis=1)

In [ ]:
tf_results = []

for tf in group0.columns:

    stat, p = ranksums(group0[tf], group1[tf])

    mean0 = group0[tf].mean()
    mean1 = group1[tf].mean()

    score = mean1 - mean0

    tf_results.append([tf, score, p])

tf_df = pd.DataFrame(tf_results, columns=["TF", "score", "pvalue"])

tf_df["FDR"] = multipletests(tf_df["pvalue"], method="fdr_bh")[1]

tf_df = tf_df.sort_values("FDR")

master_tfs = tf_df[
    (tf_df["FDR"] < 0.05) &
    (tf_df["score"].abs() > 0.5)
]

print("Master TFs:", len(master_tfs))
print(master_tfs)

In [ ]:
df = master_tfs.copy()
df = df.sort_values("score", key=np.abs, ascending=True)
high_risk_color = "#E76F51"
low_risk_color  = "#2A9D8F"

colors = [
    high_risk_color if x > 0
    else low_risk_color
    for x in df["score"]
]

n = len(df)
fig_height = max(6, n * 0.4)

fig, ax = plt.subplots(
    figsize=(8, fig_height)
)

ax.barh(
    df["TF"],
    df["score"],
    color=colors,
    height=0.6,
    edgecolor="black",
    linewidth=0.4
)

max_val = np.max(np.abs(df["score"]))

ax.set_xlim(
    -max_val * 1.05,
    max_val * 1.05
)

ax.axvline(
    0,
    color="black",
    linewidth=1
)

ax.set_xlabel("TF Activity (ULM Score)")
ax.set_ylabel("Transcription Factors")
ax.set_title(
    "Differential Transcription Factor Activity",
    pad=10
)

#for spine in ["top", "right"]:
    #ax.spines[spine].set_visible(False)

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.2
)

legend_elements = [
    Patch(
        facecolor=high_risk_color,
        edgecolor="black",
        linewidth=0.4,
        label="Higher in High-Risk Group"
    ),
    Patch(
        facecolor=low_risk_color,
        edgecolor="black",
        linewidth=0.4,
        label="Higher in Low-Risk Group"
    )
]

ax.legend(
    handles=legend_elements,
    frameon=False,
    loc="lower left",
    handlelength=1.2
)

plt.tight_layout()
plt.savefig(
    "TF_master_regulators.png",
    bbox_inches='tight'
)
plt.show()

Comparative Evaluation of Feature Representation Methods
for Survival Prediction

In [ ]:
N_COMPONENTS = 100
TOP_K = 30
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rna_final)
feature_dict = {
    "Raw": pd.DataFrame(X_scaled, index=rna_final.index),
    "PCA": pd.DataFrame(PCA(n_components=N_COMPONENTS).fit_transform(X_scaled), index=rna_final.index),
    "Autoencoder": latent_df.copy()
}

results = []

for name, feat_df in feature_dict.items():

    print(f"\n====== {name} ======")
    pvals, valid_cols = [], []

    for col in feat_df.columns:
        df = pd.DataFrame({
            col: feat_df[col],
            "time": survival["time"],
            "event": survival["recurrence"]
        }).dropna()

        try:
            cph = CoxPHFitter()
            cph.fit(df, duration_col="time", event_col="event")
            p = cph.summary.loc[col, "p"]
        except:
            p = 1

        pvals.append(p)
        valid_cols.append(col)

    fdr = multipletests(pvals, method="fdr_bh")[1]
    sig_features = [col for col, f in zip(valid_cols, fdr) if f < 0.05]

    if len(sig_features) == 0:
        sig_features = valid_cols[:TOP_K]

    sig_features = sig_features[:TOP_K]

    print(f"Selected features: {len(sig_features)}")

    X_sig = feat_df[sig_features]

    cox_df = X_sig.copy()
    cox_df["time"] = survival["time"].values
    cox_df["event"] = survival["recurrence"].values

    cph = CoxPHFitter(penalizer=0.1)
    cph.fit(cox_df, duration_col="time", event_col="event")

    risk = cph.predict_partial_hazard(cox_df)

    cindex = concordance_index(
        cox_df["time"], -risk, cox_df["event"]
    )

    kmeans = KMeans(n_clusters=2, random_state=42, n_init=20)
    clusters = kmeans.fit_predict(X_sig)

    risk_df = survival.copy()
    risk_df["cluster"] = clusters

    g0_mean = risk_df[risk_df["cluster"] == 0]["time"].mean()
    g1_mean = risk_df[risk_df["cluster"] == 1]["time"].mean()

    label_map = {0: "Low Risk", 1: "High Risk"} if g0_mean > g1_mean else {1: "Low Risk", 0: "High Risk"}

    g0 = risk_df[risk_df["cluster"] == 0]
    g1 = risk_df[risk_df["cluster"] == 1]

    lr = logrank_test(
        g0["time"], g1["time"],
        event_observed_A=g0["recurrence"],
        event_observed_B=g1["recurrence"]
    )

    p_value = lr.p_value

    kmf = KaplanMeierFitter()

    plt.figure(figsize=(6,4))

    for label in ["Low Risk", "High Risk"]:
        for c, lname in label_map.items():
            if lname == label:
                mask = risk_df["cluster"] == c

                kmf.fit(
                    durations=risk_df["time"][mask],
                    event_observed=risk_df["recurrence"][mask],
                    label=label
                )
                kmf.plot_survival_function(ci_show=False)

    plt.title(name)
    plt.xlabel("Time (Days)")
    plt.ylabel("Survival Probability")

    plt.text(0.6, 0.3, f"p = {p_value:.2e}",
             transform=plt.gca().transAxes)

    plt.grid(alpha=0.2)
    plt.tight_layout()

    plt.savefig(f"KM_{name}.png", dpi=300)
    plt.show()

    results.append({
        "Model": name,
        "Features Used": len(sig_features),
        "C-index": round(cindex, 3),
        "Log-rank p-value": "{:.2e}".format(p_value)
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(by="Log-rank p-value")

print("\n===== FINAL COMPARISON TABLE =====\n")
print(results_df.to_string(index=False))